# Docker Containerization for ML

## Learning Objectives
- Understand containerization concepts for ML
- Create optimized Dockerfiles for ML applications
- Build multi-stage Docker builds
- Use Docker Compose for local development
- Implement best practices for production containers

## Why Docker for ML?

Docker solves the "it works on my machine" problem:

| Challenge | Docker Solution |
|-----------|----------------|
| Dependency conflicts | Isolated environments |
| Environment differences | Identical containers everywhere |
| Reproducibility | Versioned container images |
| Deployment complexity | Single deployable artifact |
| Scaling | Easy horizontal scaling |

## 1. Docker Fundamentals

### Key Concepts

- **Image**: A read-only template with instructions for creating a container
- **Container**: A runnable instance of an image
- **Dockerfile**: A text file containing instructions to build an image
- **Registry**: A repository for storing and distributing images

### Docker Architecture

```
┌─────────────────────────────────────────────────────┐
│                    Docker Host                       │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐  │
│  │ Container 1 │  │ Container 2 │  │ Container 3 │  │
│  │   ML API    │  │   Worker    │  │   Redis     │  │
│  └─────────────┘  └─────────────┘  └─────────────┘  │
│  ┌─────────────────────────────────────────────────┐ │
│  │              Docker Engine                       │ │
│  └─────────────────────────────────────────────────┘ │
│  ┌─────────────────────────────────────────────────┐ │
│  │              Host Operating System               │ │
│  └─────────────────────────────────────────────────┘ │
└─────────────────────────────────────────────────────┘
```

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import os

# Create docker directory for examples
docker_dir = Path('../docker-examples')
docker_dir.mkdir(exist_ok=True)

print("Docker examples directory created")

## 2. Basic Dockerfile for ML

A simple Dockerfile for serving a scikit-learn model.

In [ ]:
basic_dockerfile = '''
# Basic ML API Dockerfile
# ======================

# Use official Python image
FROM python:3.12-slim

# Set working directory
WORKDIR /app

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PIP_NO_CACHE_DIR=1 \\
    PIP_DISABLE_PIP_VERSION_CHECK=1

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first (for caching)
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Run the application
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

# Save Dockerfile
with open(docker_dir / 'Dockerfile.basic', 'w') as f:
    f.write(basic_dockerfile)

print("Basic Dockerfile created!")
print(basic_dockerfile)

## 3. Multi-Stage Dockerfile

Multi-stage builds create smaller, more secure production images.

In [ ]:
multistage_dockerfile = '''
# Multi-Stage ML API Dockerfile
# ==============================

# =============================================================================
# Stage 1: Builder - Install dependencies and build
# =============================================================================
FROM python:3.12-slim AS builder

WORKDIR /app

# Install build dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Create virtual environment
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Copy and install requirements
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# =============================================================================
# Stage 2: Production - Minimal runtime image
# =============================================================================
FROM python:3.12-slim AS production

# Create non-root user for security
RUN groupadd --gid 1000 appuser \\
    && useradd --uid 1000 --gid 1000 -m appuser

WORKDIR /app

# Copy virtual environment from builder
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Set environment variables
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PYTHONFAULTHANDLER=1

# Copy application code
COPY --chown=appuser:appuser . .

# Switch to non-root user
USER appuser

# Expose port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1

# Run application with gunicorn for production
CMD ["gunicorn", "app:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "-b", "0.0.0.0:8000"]
'''

with open(docker_dir / 'Dockerfile.multistage', 'w') as f:
    f.write(multistage_dockerfile)

print("Multi-stage Dockerfile created!")
print("\nBenefits of multi-stage builds:")
print("  ✓ Smaller final image (no build tools)")
print("  ✓ Faster deployments")
print("  ✓ Reduced attack surface")
print("  ✓ Cleaner layer caching")

## 4. Requirements File

Pin exact versions for reproducibility.

In [ ]:
requirements_content = '''
# ML API Requirements
# Pin exact versions for reproducibility

# Web Framework
fastapi==0.109.0
uvicorn[standard]==0.27.0
gunicorn==21.2.0
pydantic==2.5.3

# ML Libraries
numpy==1.26.3
pandas==2.1.4
scikit-learn==1.3.2
joblib==1.3.2

# Monitoring & Logging
prometheus-client==0.19.0
structlog==24.1.0

# HTTP Client (for health checks)
httpx==0.26.0
'''

with open(docker_dir / 'requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Requirements file created!")
print(requirements_content)

## 5. Docker Compose for Development

Docker Compose orchestrates multiple containers for local development.

In [ ]:
docker_compose_content = '''
# Docker Compose for ML Development
# ==================================

version: "3.9"

services:
  # ML API Service
  ml-api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: ml-api
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models:ro  # Mount models read-only
      - ./logs:/app/logs
    environment:
      - MODEL_PATH=/app/models/classifier_model.joblib
      - LOG_LEVEL=INFO
      - WORKERS=4
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
    networks:
      - ml-network
    depends_on:
      - redis

  # Redis for caching predictions
  redis:
    image: redis:7-alpine
    container_name: ml-redis
    ports:
      - "6379:6379"
    volumes:
      - redis-data:/data
    command: redis-server --appendonly yes
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 5s
      retries: 3
    networks:
      - ml-network

  # Prometheus for metrics
  prometheus:
    image: prom/prometheus:latest
    container_name: ml-prometheus
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - prometheus-data:/prometheus
    networks:
      - ml-network

  # Grafana for visualization
  grafana:
    image: grafana/grafana:latest
    container_name: ml-grafana
    ports:
      - "3000:3000"
    volumes:
      - grafana-data:/var/lib/grafana
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
    networks:
      - ml-network
    depends_on:
      - prometheus

networks:
  ml-network:
    driver: bridge

volumes:
  redis-data:
  prometheus-data:
  grafana-data:
'''

with open(docker_dir / 'docker-compose.yml', 'w') as f:
    f.write(docker_compose_content)

print("Docker Compose file created!")
print("\nServices:")
print("  - ml-api: FastAPI model serving (port 8000)")
print("  - redis: Caching layer (port 6379)")
print("  - prometheus: Metrics (port 9090)")
print("  - grafana: Dashboards (port 3000)")

## 6. .dockerignore File

Exclude unnecessary files from the build context.

In [ ]:
dockerignore_content = '''
# .dockerignore - Exclude from Docker build context
# ==================================================

# Git
.git
.gitignore

# Python
__pycache__
*.py[cod]
*$py.class
*.so
.Python
venv/
env/
.venv/
*.egg-info/
.eggs/
dist/
build/

# IDE
.idea/
.vscode/
*.swp
*.swo

# Jupyter
.ipynb_checkpoints/
*.ipynb

# Testing
.pytest_cache/
.coverage
htmlcov/
tests/

# Data (large files)
data/
*.csv
*.parquet

# Docker
Dockerfile*
docker-compose*.yml
.docker/

# Documentation
docs/
*.md
README*

# Logs
logs/
*.log

# Environment
.env
.env.*
!.env.example
'''

with open(docker_dir / '.dockerignore', 'w') as f:
    f.write(dockerignore_content)

print(".dockerignore created!")
print("\nExcluded items reduce build context size and improve security.")

## 7. GPU-Enabled Dockerfile

For deep learning models that need GPU acceleration.

In [ ]:
gpu_dockerfile = '''
# GPU-Enabled ML Dockerfile
# =========================

# Use NVIDIA CUDA base image
FROM nvidia/cuda:12.1.0-cudnn8-runtime-ubuntu22.04

# Prevent interactive prompts
ENV DEBIAN_FRONTEND=noninteractive

# Install Python and dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    python3.11 \\
    python3-pip \\
    python3-venv \\
    && rm -rf /var/lib/apt/lists/* \\
    && ln -s /usr/bin/python3.11 /usr/bin/python

WORKDIR /app

# Create virtual environment
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# Install PyTorch with CUDA support
RUN pip install --no-cache-dir \\
    torch==2.1.2+cu121 \\
    torchvision==0.16.2+cu121 \\
    --index-url https://download.pytorch.org/whl/cu121

# Install other requirements
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY . .

# Set NVIDIA runtime environment
ENV NVIDIA_VISIBLE_DEVICES=all
ENV NVIDIA_DRIVER_CAPABILITIES=compute,utility

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open(docker_dir / 'Dockerfile.gpu', 'w') as f:
    f.write(gpu_dockerfile)

print("GPU Dockerfile created!")
print("\nTo run with GPU:")
print("  docker run --gpus all -p 8000:8000 ml-api-gpu")

## 8. Docker Build Commands

Common Docker commands for building and running containers.

In [ ]:
docker_commands = '''
# Docker Commands Cheatsheet
# ==========================

# Build image
docker build -t ml-api:latest .
docker build -t ml-api:v1.0.0 -f Dockerfile.multistage .

# Run container
docker run -d -p 8000:8000 --name ml-api ml-api:latest
docker run -d -p 8000:8000 -v ./models:/app/models ml-api:latest

# View logs
docker logs ml-api
docker logs -f ml-api  # Follow logs

# Execute command in container
docker exec -it ml-api bash
docker exec ml-api python -c "import torch; print(torch.cuda.is_available())"

# Container management
docker ps                    # List running containers
docker ps -a                 # List all containers
docker stop ml-api           # Stop container
docker rm ml-api             # Remove container
docker restart ml-api        # Restart container

# Image management
docker images                # List images
docker rmi ml-api:latest     # Remove image
docker system prune -a       # Clean up unused resources

# Docker Compose
docker-compose up -d         # Start all services
docker-compose down          # Stop all services
docker-compose logs -f       # Follow all logs
docker-compose ps            # List services
docker-compose build         # Rebuild services

# Push to registry
docker tag ml-api:latest registry.example.com/ml-api:latest
docker push registry.example.com/ml-api:latest
'''

print(docker_commands)

## 9. Container Optimization Tips

Best practices for production-ready containers.

In [ ]:
optimization_tips = {
    "Use slim/alpine base images": {
        "bad": "FROM python:3.12",
        "good": "FROM python:3.12-slim",
        "impact": "~500MB smaller image"
    },
    "Order layers by change frequency": {
        "bad": "COPY . . → RUN pip install",
        "good": "COPY requirements.txt → RUN pip install → COPY . .",
        "impact": "Better layer caching"
    },
    "Combine RUN commands": {
        "bad": "RUN apt-get update\nRUN apt-get install git",
        "good": "RUN apt-get update && apt-get install git && rm -rf /var/lib/apt/lists/*",
        "impact": "Fewer layers, smaller size"
    },
    "Use .dockerignore": {
        "bad": "Include everything in build context",
        "good": "Exclude .git, __pycache__, data/, tests/",
        "impact": "Faster builds, smaller context"
    },
    "Run as non-root user": {
        "bad": "Run as root (default)",
        "good": "USER appuser",
        "impact": "Better security"
    },
    "Use multi-stage builds": {
        "bad": "Include build tools in final image",
        "good": "Builder stage → Production stage",
        "impact": "60-80% smaller images"
    }
}

print("=" * 60)
print("DOCKER OPTIMIZATION BEST PRACTICES")
print("=" * 60)

for tip, details in optimization_tips.items():
    print(f"\n📌 {tip}")
    print(f"   ❌ Bad:  {details['bad']}")
    print(f"   ✅ Good: {details['good']}")
    print(f"   💡 Impact: {details['impact']}")

## 10. Summary

### Files Created

| File | Purpose |
|------|--------|
| `Dockerfile.basic` | Simple single-stage build |
| `Dockerfile.multistage` | Optimized multi-stage build |
| `Dockerfile.gpu` | GPU-enabled for deep learning |
| `docker-compose.yml` | Multi-container development |
| `requirements.txt` | Pinned Python dependencies |
| `.dockerignore` | Exclude files from build |

In [ ]:
# List created files
print("=" * 50)
print("DOCKER CONTAINERIZATION COMPLETE")
print("=" * 50)
print("\nFiles created in docker-examples/:")

for f in sorted(docker_dir.iterdir()):
    size = f.stat().st_size
    print(f"  📄 {f.name} ({size} bytes)")

print("\nQuick Start:")
print("  cd docker-examples")
print("  docker build -t ml-api .")
print("  docker run -p 8000:8000 ml-api")
print("\nWith Docker Compose:")
print("  docker-compose up -d")